# Анализ данных ЛР1

Инфляция и долгосрочная процентная ставка Германии, 1972Q2–1998Q4.

## 1. Загрузка данных

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", 15)
pd.set_option("display.precision", 6)

df = pd.read_csv("data/E6.csv")
print(f"Строк: {len(df)}, столбцов: {df.shape[1]}")
display(df.head())

Строк: 107, столбцов: 4


,year,quarter,Dp,R
0,1972,2,-0.003133,0.083
1,1972,3,0.018871,0.083
2,1972,4,0.024804,0.087
3,1973,1,0.016278,0.087
4,1973,2,0.000290,0.102


## 2. Анализ качества

`year` — год, `quarter` — квартал, `Dp` — изменение логарифма дефлятора ВВП,
`R` — долгосрочная ставка в долях единицы.

In [2]:
numeric = df.apply(pd.to_numeric, errors="coerce")
display(pd.DataFrame({
    "Тип данных": df.dtypes.astype(str),
    "Пропуски": df.isna().sum(),
    "Нечисловые значения": (df.notna() & numeric.isna()).sum(),
    "Минимум": numeric.min(),
    "Максимум": numeric.max()
}))
print("Полные дубликаты:", df.duplicated().sum())
print("Повторы периода:", df.duplicated(["year", "quarter"]).sum())
print("Категории квартала:", sorted(df["quarter"].unique().tolist()))

,Тип данных,Пропуски,Нечисловые значения,Минимум,Максимум
year,int64,0,0,1972.000000,1998.000000
quarter,int64,0,0,1.000000,4.000000
Dp,float64,0,0,-0.029195,0.043004
R,float64,0,0,0.038000,0.113000


Полные дубликаты: 0
Повторы периода: 0
Категории квартала: [1, 2, 3, 4]


Год должен быть целым и находиться в диапазоне 1972–1998, квартал — принимать значения 1–4.
Для проверки грубых ошибок масштаба используются границы: от −0,5 до 0,5 для Dp и от 0 до 1 для R.

In [3]:
valid_year = numeric["year"].between(1972, 1998) & numeric["year"].mod(1).eq(0)
valid_quarter = numeric["quarter"].isin([1, 2, 3, 4])
valid_dp = numeric["Dp"].between(-0.5, 0.5)
valid_rate = numeric["R"].between(0, 1)

display(pd.DataFrame({
    "Проверка": ["Некорректный год", "Некорректный квартал",
                 "Dp вне границ", "R вне границ"],
    "Нарушений": [(~m).sum() for m in [valid_year, valid_quarter, valid_dp, valid_rate]]
}))
display(df.loc[~(valid_year & valid_quarter & valid_dp & valid_rate)])

periods = pd.PeriodIndex(
    df["year"].astype(str) + "Q" + df["quarter"].astype(str), freq="Q"
)
missing_periods = pd.period_range("1972Q2", "1998Q4", freq="Q").difference(periods)
print("Пропущенные кварталы:", missing_periods.astype(str).tolist())

,Проверка,Нарушений
0,Некорректный год,0
1,Некорректный квартал,0
2,Dp вне границ,0
3,R вне границ,0


,year,quarter,Dp,R


Пропущенные кварталы: []


## 3. Чистка данных

In [4]:
data = df.copy()
before = len(data)
data = data.drop_duplicates()
print(f"Удаление дубликатов: {before} → {len(data)} строк; удалено {before - len(data)}.")

Удаление дубликатов: 107 → 107 строк; удалено 0.


In [5]:
before_types = data.dtypes.astype(str)
missing_before = data.isna().sum()
data = data.apply(pd.to_numeric, errors="coerce")
display(pd.DataFrame({
    "Тип до": before_types,
    "Тип после": data.dtypes.astype(str),
    "Новых пропусков": data.isna().sum() - missing_before
}))

,Тип до,Тип после,Новых пропусков
year,int64,int64,0
quarter,int64,int64,0
Dp,float64,float64,0
R,float64,float64,0


In [6]:
before = len(data)
data = data.dropna()
print(f"Удаление строк с пропусками: {before} → {len(data)}; удалено {before - len(data)}.")

Удаление строк с пропусками: 107 → 107; удалено 0.


In [7]:
before = len(data)
valid = (data["year"].between(1972, 1998) & data["year"].mod(1).eq(0)
         & data["quarter"].isin([1, 2, 3, 4]))
data = data.loc[valid].copy()
print(f"Проверка категорий: {before} → {len(data)} строк; удалено {before - len(data)}.")

Проверка категорий: 107 → 107 строк; удалено 0.


In [8]:
before = len(data)
data = data.loc[data["Dp"].between(-0.5, 0.5) & data["R"].between(0, 1)].copy()
print(f"Проверка границ: {before} → {len(data)} строк; удалено {before - len(data)}.")
print(f"Итог чистки: {len(df)} → {len(data)} строк.")

Проверка границ: 107 → 107 строк; удалено 0.
Итог чистки: 107 → 107 строк.


В данных нарушений не обнаружено, поэтому все 107 наблюдений сохранены.

## 4. Преобразование и добавление полей

Инфляция выражена в квартальных логарифмических процентах, ставка — в процентах годовых.
Дата конца квартала используется как условная календарная метка.

In [9]:
data = data.sort_values(["year", "quarter"]).reset_index(drop=True)
data["period"] = pd.PeriodIndex(
    data["year"].astype(str) + "Q" + data["quarter"].astype(str), freq="Q"
)
data["date"] = data["period"].dt.to_timestamp(how="end").dt.normalize()
data["inflation_log_pct"] = data["Dp"] * 100
data["interest_pct"] = data["R"] * 100
display(data.head())

,year,quarter,Dp,R,period,date,inflation_log_pct,interest_pct
0,1972,2,-0.003133,0.083,1972Q2,1972-06-30,-0.313258,8.3
1,1972,3,0.018871,0.083,1972Q3,1972-09-30,1.887130,8.3
2,1972,4,0.024804,0.087,1972Q4,1972-12-31,2.480360,8.7
3,1973,1,0.016278,0.087,1973Q1,1973-03-31,1.627760,8.7
4,1973,2,0.000290,0.102,1973Q2,1973-06-30,0.028968,10.2


## 5. Статистический анализ
### Типы данных

In [10]:
display(data.dtypes.astype(str).to_frame("Тип данных"))

,Тип данных
year,int64
quarter,int64
Dp,float64
R,float64
period,period[Q-DEC]
date,datetime64[ns]
inflation_log_pct,float64
interest_pct,float64


### Меры центральной тенденции и разброса

Для экономических показателей рассчитываются среднее, медиана, мода, минимум, максимум,
размах, выборочная дисперсия и стандартное отклонение. Год и квартал — календарные признаки.

In [11]:
columns = ["Dp", "R", "inflation_log_pct", "interest_pct"]
stats = data[columns].agg(["count", "min", "max", "mean", "median", "var", "std"]).T
stats.columns = ["Количество", "Минимум", "Максимум", "Среднее", "Медиана",
                 "Дисперсия", "Станд. отклонение"]
stats["Размах"] = stats["Максимум"] - stats["Минимум"]
stats["Q1"] = data[columns].quantile(0.25)
stats["Q3"] = data[columns].quantile(0.75)
stats["МКР"] = stats["Q3"] - stats["Q1"]
display(stats)

modes = []
for column in columns:
    counts = data[column].value_counts()
    values = counts[counts.eq(counts.max())].index.sort_values().tolist()
    modes.append({"Поле": column, "Мода": values if counts.max() > 1 else "Не выделяется",
                  "Частота": counts.max()})
display(pd.DataFrame(modes).set_index("Поле"))

,Количество,Минимум,Максимум,Среднее,Медиана,Дисперсия,Станд. отклонение,Размах,Q1,Q3,МКР
Dp,107.0,-0.029195,0.043004,0.008397,0.008706,0.000356,0.018876,0.072199,-0.004265,0.02391,0.028175
R,107.0,0.038000,0.113000,0.074579,0.074000,0.000273,0.016511,0.075000,0.060500,0.08600,0.025500
inflation_log_pct,107.0,-2.919480,4.300400,0.839731,0.870609,3.562974,1.887584,7.219880,-0.426459,2.39103,2.817489
interest_pct,107.0,3.800000,11.300000,7.457944,7.400000,2.726045,1.651074,7.500000,6.050000,8.60000,2.550000


,Мода,Частота
Поле,,
Dp,Не выделяется,1
R,[0.06],7
inflation_log_pct,Не выделяется,1
interest_pct,[6.0],7


### Выбросы

Границы Тьюки: Q1 − 1,5 × МКР и Q3 + 1,5 × МКР. Значения строго за границами считаются выбросами.

In [12]:
bounds = stats[["Q1", "Q3", "МКР"]].copy()
bounds["Нижняя граница"] = bounds["Q1"] - 1.5 * bounds["МКР"]
bounds["Верхняя граница"] = bounds["Q3"] + 1.5 * bounds["МКР"]
outliers = []

for column in columns:
    mask = (data[column] < bounds.loc[column, "Нижняя граница"]) | (
        data[column] > bounds.loc[column, "Верхняя граница"])
    bounds.loc[column, "Выбросов"] = int(mask.sum())
    for _, row in data.loc[mask].iterrows():
        outliers.append({"Период": row["period"], "Поле": column, "Значение": row[column]})

bounds["Выбросов"] = bounds["Выбросов"].astype(int)
display(bounds)
display(pd.DataFrame(outliers, columns=["Период", "Поле", "Значение"]))
print("Всего выбросов:", len(outliers))

,Q1,Q3,МКР,Нижняя граница,Верхняя граница,Выбросов
Dp,-0.004265,0.02391,0.028175,-0.046527,0.066173,0
R,0.060500,0.08600,0.025500,0.022250,0.124250,0
inflation_log_pct,-0.426459,2.39103,2.817489,-4.652694,6.617264,0
interest_pct,6.050000,8.60000,2.550000,2.225000,12.425000,0


,Период,Поле,Значение


Всего выбросов: 0


### Частотные ряды

Для года и квартала используются точные частоты. Для количественных показателей — равные интервалы
по правилу Стерджеса. Интервалы включают левую границу; последний включает и правую.

In [13]:
for column in ["year", "quarter"]:
    frequencies = data[column].value_counts().sort_index().rename("Частота").to_frame()
    frequencies["Доля"] = frequencies["Частота"] / len(data)
    print(column)
    display(frequencies)

year


,Частота,Доля
year,,
1972,3,0.028037
1973,4,0.037383
1974,4,0.037383
1975,4,0.037383
1976,4,0.037383
1977,4,0.037383
1978,4,0.037383
1979,4,0.037383
1980,4,0.037383


quarter


,Частота,Доля
quarter,,
1,26,0.242991
2,27,0.252336
3,27,0.252336
4,27,0.252336


In [14]:
k = int(np.ceil(1 + np.log2(len(data))))
print("Количество интервалов:", k)

for column in columns:
    frequencies, edges = np.histogram(data[column], bins=k)
    intervals = [f"[{edges[i]:.6f}; {edges[i + 1]:.6f}{']' if i == k - 1 else ')'}"
                 for i in range(k)]
    table = pd.DataFrame({
        "Интервал": intervals,
        "Частота": frequencies,
        "Доля": frequencies / len(data),
        "Накопленная частота": frequencies.cumsum()
    })
    print(column)
    display(table)
    print("Сумма частот:", table["Частота"].sum())

Количество интервалов: 8
Dp


,Интервал,Частота,Доля,Накопленная частота
0,[-0.029195; -0.020170),8,0.074766,8
1,[-0.020170; -0.011145),14,0.130841,22
2,[-0.011145; -0.002120),8,0.074766,30
3,[-0.002120; 0.006905),20,0.186916,50
4,[0.006905; 0.015929),17,0.158879,67
5,[0.015929; 0.024954),18,0.168224,85
6,[0.024954; 0.033979),9,0.084112,94
7,[0.033979; 0.043004],13,0.121495,107


Сумма частот: 107
R


,Интервал,Частота,Доля,Накопленная частота
0,[0.038000; 0.047375),4,0.037383,4
1,[0.047375; 0.056750),12,0.112150,16
2,[0.056750; 0.066125),24,0.224299,40
3,[0.066125; 0.075500),15,0.140187,55
4,[0.075500; 0.084875),23,0.214953,78
5,[0.084875; 0.094250),16,0.149533,94
6,[0.094250; 0.103625),7,0.065421,101
7,[0.103625; 0.113000],6,0.056075,107


Сумма частот: 107
inflation_log_pct


,Интервал,Частота,Доля,Накопленная частота
0,[-2.919480; -2.016995),8,0.074766,8
1,[-2.016995; -1.114510),14,0.130841,22
2,[-1.114510; -0.212025),8,0.074766,30
3,[-0.212025; 0.690460),20,0.186916,50
4,[0.690460; 1.592945),17,0.158879,67
5,[1.592945; 2.495430),18,0.168224,85
6,[2.495430; 3.397915),9,0.084112,94
7,[3.397915; 4.300400],13,0.121495,107


Сумма частот: 107
interest_pct


,Интервал,Частота,Доля,Накопленная частота
0,[3.800000; 4.737500),4,0.037383,4
1,[4.737500; 5.675000),12,0.112150,16
2,[5.675000; 6.612500),24,0.224299,40
3,[6.612500; 7.550000),15,0.140187,55
4,[7.550000; 8.487500),23,0.214953,78
5,[8.487500; 9.425000),16,0.149533,94
6,[9.425000; 10.362500),7,0.065421,101
7,[10.362500; 11.300000],6,0.056075,107


Сумма частот: 107
